<a href="https://colab.research.google.com/github/ga426553-sudo/Classification-of-Breast-Cancer-Subtypes-machine-learning---CuMiDa-22820/blob/main/No_5_Decisiontree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Quinto código - Cáncer de Mama 🌸

**Implementación de Decision tree para Clasificación de Cáncer de Mama**

###Importar librerías necesarias 🌺

In [ ]:
# ============================================
# CÓDIGO 5: DECISION TREE Y EVALUACIÓN
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import pickle
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print("="*60)
print("🌳 CÓDIGO 6: EVALUACIÓN DE DECISION TREE")
print("="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🌳 CÓDIGO 6: EVALUACIÓN DE DECISION TREE


###Cargar datos 🌺

In [ ]:
# 1. CARGAR DATOS
# ================
print(f"\n📂 Cargando datos del Código 2...")

with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/subsets.pkl', 'rb') as f:
    subsets = pickle.load(f)

y_train = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_train_final.npy')
y_test = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_test_final.npy')

print(f"✅ Datos cargados:")
print(f"   Subconjuntos: {list(subsets.keys())}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test: {y_test.shape}")


📂 Cargando datos del Código 2...
✅ Datos cargados:
   Subconjuntos: ['EO', 'BBA', 'CSA', 'RDA', 'GA']
   y_train: (206,)
   y_test: (28,)


### Configuración del decision tree 🌺

In [ ]:
# 2. CONFIGURAR DECISION TREE (parámetros típicos del artículo)
# ==============================================================
dt_params = {
    'criterion': 'gini',        # o 'entropy'
    'max_depth': 4,             # igual que en XGBoost para comparación
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'random_state': 42
}

dt_model = DecisionTreeClassifier(**dt_params)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print(f"\n⚙️ Configuración de Decision Tree:")
for key, value in dt_params.items():
    print(f"   {key}: {value}")


⚙️ Configuración de Decision Tree:
   criterion: gini
   max_depth: 4
   min_samples_split: 2
   min_samples_leaf: 1
   random_state: 42


###Evaluación de cada subconjunto 🌺

In [ ]:
# 3. EVALUAR CADA SUBCONJUNTO
# ============================
results = {}

for name in subsets.keys():
    print(f"\n{'─'*40}")
    print(f"🔍 Subconjunto: {name} - Genes: {subsets[name]}")

    X_train_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/train_{name}.npy')
    X_test_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/test_{name}.npy')

    # Validación cruzada (10-fold)
    cv_acc = cross_val_score(dt_model, X_train_subset, y_train, cv=cv, scoring='accuracy')
    cv_f1 = cross_val_score(dt_model, X_train_subset, y_train, cv=cv, scoring='f1')
    cv_auc = cross_val_score(dt_model, X_train_subset, y_train, cv=cv, scoring='roc_auc')

    # Entrenar modelo completo
    dt_model.fit(X_train_subset, y_train)

    # Predicciones
    y_pred = dt_model.predict(X_test_subset)
    y_proba = dt_model.predict_proba(X_test_subset)[:, 1]

    # Métricas en test
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_proba)

    # Importancia de características
    feature_importance = pd.DataFrame({
        'gene': subsets[name],
        'importance': dt_model.feature_importances_
    }).sort_values('importance', ascending=False)

    results[name] = {
        'cv_accuracy': f"{cv_acc.mean():.4f} ± {cv_acc.std():.4f}",
        'cv_f1': f"{cv_f1.mean():.4f} ± {cv_f1.std():.4f}",
        'cv_auc': f"{cv_auc.mean():.4f} ± {cv_auc.std():.4f}",
        'test_accuracy': test_acc,
        'test_f1': test_f1,
        'test_auc': test_auc,
        'genes': subsets[name],
        'feature_importance': feature_importance,
        'y_pred': y_pred,
        'y_proba': y_proba
    }

    print(f"\n   Resultados:")
    print(f"   📊 Validación Cruzada (10-fold):")
    print(f"      Accuracy: {results[name]['cv_accuracy']}")
    print(f"      F1-Score: {results[name]['cv_f1']}")
    print(f"      AUC: {results[name]['cv_auc']}")
    print(f"\n   📈 Test:")
    print(f"      Accuracy: {test_acc:.4f}")
    print(f"      F1-Score: {test_f1:.4f}")
    print(f"      AUC: {test_auc:.4f}")
    print(f"\n   🔬 Importancia de genes:")
    print(feature_importance.to_string(index=False))



────────────────────────────────────────
🔍 Subconjunto: EO - Genes: ['NM_138957', 'NM_152426', 'NM_001008493']

   Resultados:
   📊 Validación Cruzada (10-fold):
      Accuracy: 0.9229 ± 0.0430
      F1-Score: 0.9224 ± 0.0423
      AUC: 0.9463 ± 0.0440

   📈 Test:
      Accuracy: 0.9643
      F1-Score: 0.9811
      AUC: 0.7500

   🔬 Importancia de genes:
        gene  importance
   NM_138957    0.771291
NM_001008493    0.145526
   NM_152426    0.083183

────────────────────────────────────────
🔍 Subconjunto: BBA - Genes: ['NM_152426', 'NM_138957']

   Resultados:
   📊 Validación Cruzada (10-fold):
      Accuracy: 0.8990 ± 0.0720
      F1-Score: 0.8973 ± 0.0721
      AUC: 0.9203 ± 0.0656

   📈 Test:
      Accuracy: 0.9286
      F1-Score: 0.9615
      AUC: 0.7308

   🔬 Importancia de genes:
     gene  importance
NM_138957    0.823476
NM_152426    0.176524

────────────────────────────────────────
🔍 Subconjunto: CSA - Genes: ['BC016934', 'NM_006579']

   Resultados:
   📊 Validación Cruza

###Resultados comparativos 🌺

In [ ]:
# 4. MOSTRAR RESULTADOS COMPARATIVOS
# ===================================
print(f"\n{'='*60}")
print("📊 RESULTADOS FINALES - DECISION TREE")
print('='*60)

comparison_df = pd.DataFrame({
    'Subset': results.keys(),
    'Genes': [', '.join(r['genes']) for r in results.values()],
    'Test_Accuracy': [f"{r['test_accuracy']:.4f}" for r in results.values()],
    'Test_F1': [f"{r['test_f1']:.4f}" for r in results.values()],
    'Test_AUC': [f"{r['test_auc']:.4f}" for r in results.values()],
    'CV_Accuracy': [r['cv_accuracy'] for r in results.values()]
})

print("\n📋 Tabla comparativa:")
print(comparison_df.to_string(index=False))


📊 RESULTADOS FINALES - DECISION TREE

📋 Tabla comparativa:
Subset                              Genes Test_Accuracy Test_F1 Test_AUC     CV_Accuracy
    EO NM_138957, NM_152426, NM_001008493        0.9643  0.9811   0.7500 0.9229 ± 0.0430
   BBA               NM_152426, NM_138957        0.9286  0.9615   0.7308 0.8990 ± 0.0720
   CSA                BC016934, NM_006579        0.8571  0.9200   0.7115 0.9271 ± 0.0452
   RDA                           BC016934        0.5357  0.6829   0.5096 0.7952 ± 0.0808
    GA                          NM_152426        0.8571  0.9167   0.9038 0.7283 ± 0.0724


In [ ]:
# ============================================
# BASELINE: Decision Tree con TODOS los genes
# ============================================

print("\n" + "="*60)
print("📊 BASELINE - Decision Tree con TODOS LOS GENES")
print("="*60)

X_train_all = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/X_train_balanced.npy')
X_test_all = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/X_test_clean.npy')

print(f"Datos baseline: {X_train_all.shape[1]} genes")

dt_baseline = DecisionTreeClassifier(
    criterion='gini',
    max_depth=4,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

cv_baseline = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print("\n🔄 Ejecutando validación cruzada (10-fold)...")
print("⚠️ Esto puede tomar varios minutos...")

cv_acc = cross_val_score(dt_baseline, X_train_all, y_train, cv=cv_baseline, scoring='accuracy')
cv_f1 = cross_val_score(dt_baseline, X_train_all, y_train, cv=cv_baseline, scoring='f1')
cv_auc = cross_val_score(dt_baseline, X_train_all, y_train, cv=cv_baseline, scoring='roc_auc')

print(f"\n📊 Resultados Baseline Decision Tree:")
print(f"   CV Accuracy: {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
print(f"   CV F1-Score: {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")
print(f"   CV AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")

dt_baseline.fit(X_train_all, y_train)
y_pred = dt_baseline.predict(X_test_all)
y_proba = dt_baseline.predict_proba(X_test_all)[:, 1]

test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_proba)

print(f"\n📊 Test Baseline Decision Tree:")
print(f"   Test Accuracy: {test_acc:.4f}")
print(f"   Test F1-Score: {test_f1:.4f}")
print(f"   Test AUC: {test_auc:.4f}")

baseline_results = {
    'cv_accuracy': f"{cv_acc.mean():.4f} ± {cv_acc.std():.4f}",
    'cv_f1': f"{cv_f1.mean():.4f} ± {cv_f1.std():.4f}",
    'cv_auc': f"{cv_auc.mean():.4f} ± {cv_auc.std():.4f}",
    'test_accuracy': test_acc,
    'test_f1': test_f1,
    'test_auc': test_auc
}

with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/baseline_dt.pkl', 'wb') as f:
    pickle.dump(baseline_results, f)

print("\n✅ Baseline Decision Tree guardado")


📊 BASELINE - Decision Tree con TODOS LOS GENES
Datos baseline: 31215 genes

🔄 Ejecutando validación cruzada (10-fold)...
⚠️ Esto puede tomar varios minutos...

📊 Resultados Baseline Decision Tree:
   CV Accuracy: 0.9952 ± 0.0143
   CV F1-Score: 0.9952 ± 0.0143
   CV AUC: 0.9955 ± 0.0136

📊 Test Baseline Decision Tree:
   Test Accuracy: 0.9643
   Test F1-Score: 0.9804
   Test AUC: 0.9808

✅ Baseline Decision Tree guardado


### Evaluación con Repeated Stratified K-Fold 🌺

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
import numpy as np
import pickle

# Cargar datos (si no los tienes ya en memoria)
# Asumiendo que 'subsets' y 'y_train' ya están cargados de celdas anteriores
# with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/subsets.pkl', 'rb') as f:
#     subsets = pickle.load(f)
# y_train = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_train_final.npy')

# Parámetros del Decision Tree (igual que en tu código)
dt = DecisionTreeClassifier(criterion='gini', max_depth=4, min_samples_split=2, min_samples_leaf=1, random_state=42)

# CV repetida: 10 folds, 5 repeticiones (puedes usar 5 o 10 repeticiones)
rkf = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=42)

print("Resultados de Decision Tree (CV repetida 10x5):\n")
for name in subsets.keys():
    X_train_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/train_{name}.npy')
    acc = cross_val_score(dt, X_train_subset, y_train, cv=rkf, scoring='accuracy')
    f1 = cross_val_score(dt, X_train_subset, y_train, cv=rkf, scoring='f1')
    auc = cross_val_score(dt, X_train_subset, y_train, cv=rkf, scoring='roc_auc')
    print(f"{name}: Acc = {acc.mean():.4f} \u00b1 {acc.std():.4f}, "
          f"F1 = {f1.mean():.4f} \u00b1 {f1.std():.4f}, "
          f"AUC = {auc.mean():.4f} \u00b1 {auc.std():.4f}")

# Baseline: todos los genes (X_train_balanced.npy)
X_train_all = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/X_train_balanced.npy')
acc_all = cross_val_score(dt, X_train_all, y_train, cv=rkf, scoring='accuracy')
f1_all = cross_val_score(dt, X_train_all, y_train, cv=rkf, scoring='f1')
auc_all = cross_val_score(dt, X_train_all, y_train, cv=rkf, scoring='roc_auc')
print(f"\nBaseline (todos los genes): Acc = {acc_all.mean():.4f} \u00b1 {acc_all.std():.4f}, "
      f"F1 = {f1_all.mean():.4f} \u00b1 {f1_all.std():.4f}, "
      f"AUC = {auc_all.mean():.4f} \u00b1 {auc_all.std():.4f}")

Resultados de Decision Tree (CV repetida 10x5):

EO: Acc = 0.9263 ± 0.0459, F1 = 0.9247 ± 0.0472, AUC = 0.9481 ± 0.0460
BBA: Acc = 0.9080 ± 0.0576, F1 = 0.9066 ± 0.0583, AUC = 0.9228 ± 0.0604
CSA: Acc = 0.9303 ± 0.0484, F1 = 0.9274 ± 0.0503, AUC = 0.9516 ± 0.0475
RDA: Acc = 0.8067 ± 0.0666, F1 = 0.7762 ± 0.0841, AUC = 0.8192 ± 0.0817
GA: Acc = 0.7250 ± 0.1043, F1 = 0.6865 ± 0.1528, AUC = 0.8085 ± 0.0912

Baseline (todos los genes): Acc = 0.9923 ± 0.0242, F1 = 0.9919 ± 0.0261, AUC = 0.9925 ± 0.0232
